# AMANAH v0.2 — Update the EXISTING Hugging Face Endpoint

CPU-only control notebook. **No training, no new endpoint, no new subscription, no new credits.**

It updates the already-existing endpoint `semantic-integrity-v0-1-cpu` to the validated Hugging Face revision `v0.2-validated`, verifies the deployed commit, runs live release gates (including Qur'an 35:28), prints the unchanged server integration values, then scales the existing endpoint to zero to minimize idle billing.


In [ ]:
# 1) Authenticate and resolve the EXACT validated model revision
import importlib.util, json, subprocess, sys, time
from pathlib import Path

if importlib.util.find_spec('huggingface_hub') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.35,<2'],check=True)

from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

hf_token=userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('HF_TOKEN is missing from Colab Secrets. Use the existing token; do not create or buy anything new.')

api=HfApi(token=hf_token)
identity=api.whoami()
namespace=identity.get('name')
assert namespace, identity

MODEL_REPO=f'{namespace}/semantic-integrity-v0-1'
TARGET_BRANCH='v0.2-validated'
ENDPOINT_NAME='semantic-integrity-v0-1-cpu'

target_info=api.model_info(repo_id=MODEL_REPO,revision=TARGET_BRANCH)
target_sha=target_info.sha
assert target_sha, 'Could not resolve v0.2-validated commit SHA'

print('HF account:',namespace)
print('Model repo:',MODEL_REPO)
print('Target branch:',TARGET_BRANCH)
print('Target commit:',target_sha)


In [ ]:
# 2) Locate the EXISTING endpoint only — never create a new one
endpoints={ep.name:ep for ep in api.list_inference_endpoints(namespace=namespace)}
if ENDPOINT_NAME not in endpoints:
    raise RuntimeError(
        f'Existing endpoint {ENDPOINT_NAME!r} was not found. '
        'This notebook intentionally refuses to create a new endpoint or incur a new deployment.'
    )

endpoint=endpoints[ENDPOINT_NAME]
original_url=endpoint.url
original_repository=endpoint.repository
original_revision=getattr(endpoint,'revision',None)

print('Existing endpoint: FOUND')
print('Status:',endpoint.status)
print('URL:',original_url)
print('Repository:',original_repository)
print('Current revision:',original_revision)
assert original_repository==MODEL_REPO, f'Endpoint points to unexpected repository: {original_repository}'


In [ ]:
# 3) Update ONLY the repository revision; preserve hardware, URL and API contract
if str(original_revision)==str(target_sha):
    print('Endpoint already targets the validated commit; no revision update needed.')
else:
    print('Updating existing endpoint to validated commit...')
    endpoint.update(repository=MODEL_REPO,revision=target_sha)

print('Waiting for existing endpoint to become ready...')
endpoint.wait(timeout=1200,refresh_every=10)

# Refresh endpoint state from HF after deployment.
endpoint=next(ep for ep in api.list_inference_endpoints(namespace=namespace) if ep.name==ENDPOINT_NAME)
print('Endpoint status:',endpoint.status)
print('Endpoint URL:',endpoint.url)
print('Deployed revision:',endpoint.revision)
assert endpoint.repository==MODEL_REPO
assert str(endpoint.revision)==str(target_sha), (endpoint.revision,target_sha)
if original_url:
    assert endpoint.url==original_url, f'Endpoint URL unexpectedly changed: {original_url} -> {endpoint.url}'
print('ENDPOINT REVISION UPDATE: PASS')


In [ ]:
# 4) Load the reference store from the SAME validated commit
reference_path=hf_hub_download(
    repo_id=MODEL_REPO,
    filename='reference_store.json',
    repo_type='model',
    revision=target_sha,
    token=hf_token,
)
reference_store=json.loads(Path(reference_path).read_text(encoding='utf-8'))
row3528=next(x for x in reference_store['ayahs'] if x['ayah_id']=='35:28')
row22=next(x for x in reference_store['ayahs'] if x['ayah_id']=='2:2')
print('Reference store: PASS')
print('35:28 references:',len(row3528.get('references_en',[])))


In [ ]:
# 5) Live helper with cold-start allowance
import requests

HEADERS={
    'Authorization':f'Bearer {hf_token}',
    'Content-Type':'application/json',
    'X-Scale-Up-Timeout':'600',
}

def live_check(inputs,timeout=600,expected_reference_status='verified'):
    r=requests.post(endpoint.url,headers=HEADERS,json={'inputs':inputs},timeout=timeout)
    print('HTTP:',r.status_code)
    if r.status_code!=200:
        print(r.text[:5000])
    r.raise_for_status()
    result=r.json()
    required={'decision','integrity_score','severity','confidence','drifts','needs_human_review','model_version','reference_status'}
    missing=required-set(result)
    assert not missing,f'Missing response keys: {sorted(missing)}'
    assert result['model_version']=='amanah-semantic-integrity-v0.2',result
    if expected_reference_status is not None:
        assert result['reference_status']==expected_reference_status,result
    return result

print('Live helper ready:',endpoint.url)


In [ ]:
# 6) Mandatory LIVE Qur'an 35:28 release gate — wrong agency must NOT pass
bad3528=live_check({
    'source_type':'quran',
    'source_ar':row3528['source_ar'],
    'candidate_en':'And among people and moving creatures and grazing livestock are likewise various colors. Only Allah fears the knowledgeable among His servants. Indeed, Allah is Exalted in Might and Forgiving.',
    'ayah_id':'35:28',
})
print(json.dumps(bad3528,ensure_ascii=False,indent=2))
assert bad3528['decision'] in {'REVIEW','CRITICAL'},bad3528
assert any(d.get('label')=='AGENCY_SHIFT' for d in bad3528.get('drifts',[])),bad3528
assert bad3528['needs_human_review'] is True,bad3528
print('LIVE 35:28 WRONG-AGENCY GATE: PASS')


In [ ]:
# 7) LIVE 35:28 correct-direction control — must NOT raise AGENCY_SHIFT
good3528=live_check({
    'source_type':'quran',
    'source_ar':row3528['source_ar'],
    'candidate_en':'And among people and moving creatures and grazing livestock are various colors similarly. Only those fear Allah, from among His servants, who have knowledge. Indeed, Allah is Exalted in Might and Forgiving.',
    'ayah_id':'35:28',
})
print(json.dumps(good3528,ensure_ascii=False,indent=2))
assert not any(d.get('label')=='AGENCY_SHIFT' for d in good3528.get('drifts',[])),good3528
assert good3528['decision']!='CRITICAL' or not any(d.get('label')=='AGENCY_SHIFT' for d in good3528.get('drifts',[])),good3528
print('LIVE 35:28 CORRECT-DIRECTION CONTROL: PASS')


In [ ]:
# 8) LIVE 2:2 semantic-negation control — 'beyond doubt' must not trigger NEGATION_FLIP
case22=live_check({
    'source_type':'quran',
    'source_ar':row22['source_ar'],
    'candidate_en':'This is the Book, beyond doubt, a guidance for the mindful.',
    'ayah_id':'2:2',
})
print(json.dumps(case22,ensure_ascii=False,indent=2))
assert not any(d.get('label')=='NEGATION_FLIP' for d in case22.get('drifts',[])),case22
print('LIVE 2:2 NEGATION-EQUIVALENCE GATE: PASS')


In [ ]:
# 9) LIVE 2:2 true-negation control — explicit denial of the Book identity must be caught
true22=live_check({
    'source_type':'quran',
    'source_ar':row22['source_ar'],
    'candidate_en':'This is not the Book, a guidance for the mindful.',
    'ayah_id':'2:2',
})
print(json.dumps(true22,ensure_ascii=False,indent=2))
assert any(d.get('label')=='NEGATION_FLIP' for d in true22.get('drifts',[])),true22
assert true22['decision'] in {'REVIEW','CRITICAL'},true22
assert true22['needs_human_review'] is True,true22
print('LIVE 2:2 TRUE-NEGATION GATE: PASS')


In [ ]:
# 10) Verify source-mismatch safety: wrong Arabic must fail closed
mismatch=live_check({
    'source_type':'quran',
    'source_ar':'نص عربي غير مطابق للمصحف',
    'candidate_en':'This is only a safety test.',
    'ayah_id':'35:28',
}, expected_reference_status='source_mismatch')
print(json.dumps(mismatch,ensure_ascii=False,indent=2))
assert mismatch['decision']=='ABSTAIN',mismatch
assert mismatch['needs_human_review'] is True,mismatch
assert mismatch['reference_status']=='source_mismatch',mismatch
print('LIVE SOURCE-MISMATCH FAIL-CLOSED GATE: PASS')


In [ ]:
# 11) Final handoff — do not expose the token
handoff={
    'endpoint_name':ENDPOINT_NAME,
    'endpoint_url':endpoint.url,
    'repository':MODEL_REPO,
    'validated_branch':TARGET_BRANCH,
    'validated_commit':target_sha,
    'model_version':'amanah-semantic-integrity-v0.2',
    'live_35_28_wrong_agency':'PASS',
    'live_35_28_correct_control':'PASS',
    'live_2_2_negation_control':'PASS',
    'live_2_2_true_negation':'PASS',
    'live_source_mismatch':'PASS',
}
Path('/content/AMANAH_V02_LIVE_HANDOFF.json').write_text(json.dumps(handoff,ensure_ascii=False,indent=2),encoding='utf-8')

print(json.dumps(handoff,ensure_ascii=False,indent=2))
print('\nSERVER-SIDE SETTINGS FOR THE TEAM')
print('AMANAH_ML_URL='+endpoint.url)
print('AMANAH_ML_TOKEN=<keep the existing server-side Hugging Face token; never expose it in frontend code>')
print('AMANAH transport=hf')
print('\nALL LIVE V0.2 RELEASE GATES: PASS')


In [ ]:
# 12) Minimize idle cost on the EXISTING endpoint
# Hugging Face scale-to-zero keeps the endpoint and URL; the next request wakes it with a cold-start delay.
try:
    endpoint.scale_to_zero()
    print('Existing endpoint scaled to zero after validation: PASS')
    print('No idle replica should remain running. The next website request will wake it automatically.')
except Exception as exc:
    print('WARNING: automatic scale-to-zero call was not completed:',type(exc).__name__,str(exc)[:500])
    print('The deployment is still validated; check the existing endpoint autoscaling settings manually.')
